# Self-Supervised Nuclei Segmentation: encoder architecture experiments

Earlier exploration of the same self-supervised setup (scale-wise triplet + counting-ranking pretext tasks) with three encoders:
- a **U-Net encoder**,
- a **custom CNN embedding network**,
- a **ResNet-101** backbone (ImageNet-pretrained, via `torch.hub`).

Trained on **MoNuSeg 2018** and tested on a subset of the Kaggle Data Science Bowl 2018 data. Outputs are cleared to keep the file small.

In [ ]:
#GPU count and name
!nvidia-smi -L

In [ ]:
from torch.utils.data import DataLoader, Dataset
import torch
import os
from PIL import Image
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import torch.utils.data as data
import torchvision
from torchvision import datasets
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
import matplotlib.pyplot as plt
from torch.utils.data.sampler import SubsetRandomSampler
from tqdm import tqdm
import cv2
from google.colab import drive
drive.mount('/content/drive')

## Loading Kaggle Data

Original Data

In [ ]:
# # ! pip install -q kaggle
# # from google.colab import files
# # files.upload()
# # ! mkdir ~/.kaggle
# # ! cp kaggle.json ~/.kaggle/
# # ! chmod 600 ~/.kaggle/kaggle.json
# # ! kaggle datasets list
# # # guidance https://www.kaggle.com/general/74235

# ! kaggle competitions download -c data-science-bowl-2018
# ! mkdir Data
# ! mkdir Data/stage1_test
# ! mkdir Data/stage1_train
# ! unzip stage1_train.zip -d Data/stage1_train
# ! unzip stage1_test.zip -d Data/stage1_test

# dataset = 'Kaggle'

Processed Data

In [ ]:
!mkdir Tmp
!mkdir Tmp/KaggleTrainingData
!unzip  drive/My\ Drive/Kaggle/Data.zip -d Tmp/KaggleTrainingData
!mkdir Data
!mkdir Data/KaggleTrainingData
!mkdir Data/KaggleTrainingData/Images
!mkdir Data/KaggleTrainingData/Images/train/
!mkdir Data/KaggleTrainingData/Labels
!mkdir Data/KaggleTrainingData/Labels/train/

!mv Tmp/KaggleTrainingData/Data/Images/ Data/KaggleTrainingData/Images/train/
!mv Tmp/KaggleTrainingData/Data/Labels/ Data/KaggleTrainingData/Labels/train/
!rm -r Tmp

dataset = 'Kaggle'

## Loading MoNuSeg Data

In [ ]:
!mkdir Tmp
!mkdir Tmp/MoNuSegTrainingData
!mkdir Tmp/MoNuSegTestData
!unzip  drive/My\ Drive/MoNu/MoNuSegTrainingData.zip -d Tmp/MoNuSegTrainingData
!unzip  drive/My\ Drive/MoNu/MoNuSegTestData.zip -d Tmp/MoNuSegTestData
!mkdir Data
!mkdir Data/MoNuSegTrainingData
!mkdir Data/MoNuSegTrainingData/Images
!mkdir Data/MoNuSegTrainingData/Annotations
!mkdir Data/MoNuSegTestData
!mkdir Data/MoNuSegTestData/Images
!mkdir Data/MoNuSegTestData/Annotations
!mv Tmp/MoNuSegTrainingData/MoNuSeg\ Training\ Data/Tissue\ Images/* Data/MoNuSegTrainingData/Images/
!mv Tmp/MoNuSegTrainingData/MoNuSeg\ Training\ Data/Annotations/ Data/MoNuSegTrainingData/
!mv Tmp/MoNuSegTestData/MoNuSegTestData/*.tif  Data/MoNuSegTestData/Images
!mv Tmp/MoNuSegTestData/MoNuSegTestData/*.xml  Data/MoNuSegTestData/Annotations
!rm -r Tmp

dataset = 'MoNuSeg'

## Data Loader

In [ ]:
class MyDataset(Dataset):
    def __init__(self, dataset='MoNuSeg', train=True):
        if dataset == 'MoNuSeg':
            self.TRAIN_DATA_PATH = "/content/Data/MoNuSegTrainingData/"
            self.TEST_DATA_PATH = "/content/Data/MoNuSegTestData/"
        elif dataset == 'Kaggle':
            self.TRAIN_DATA_PATH = "/content/Data/KaggleTrainingData/Images/"
            self.TEST_DATA_PATH = "/content/Data/KaggleTrainingData/" ###############################################################
        self.train_data = torchvision.datasets.ImageFolder(root=self.TRAIN_DATA_PATH , )
        self.test_data = torchvision.datasets.ImageFolder(root=self.TEST_DATA_PATH)

    # returns 3 images
    def getTriplet(self, image):
        resize = transforms.Resize(size=(512, 512))
        image = resize(image)

        # Random crop, anchor
        i, j, h, w = transforms.RandomCrop.get_params(image, [384, 384])
        # print(i, j, w, h)
        anchor = TF.crop(image, i, j, h, w)

        # shift, positive
        i_shift = np.random.randint(-50, 50)  # random or fix?
        j_shift = np.random.randint(-50, 50)  # random or fix?

        new_i = i + i_shift
        new_j = j + j_shift
        if new_i < 0 or new_i + w > 512:
            new_i = i - i_shift
        if new_j < 0 or new_j + h > 512:
            new_j = j - j_shift
        pos = TF.crop(image, new_i, new_j, w, h)

        # negative
        scale = np.random.choice([6, 7, 8])
        i, j, h, w = transforms.RandomCrop.get_params(pos, [2 ** scale, 2 ** scale])
        neg = TF.crop(pos, i, j, h, w)
        resize = transforms.Resize(size=(384, 384))  # quality lowers a lot specially for 64*64 size
        neg = resize(neg)
        return anchor, pos, neg

    # returns tensor
    def train_getitem(self, index):
        train_item = self.train_data[index]
        anchor, pos, neg = self.getTriplet(train_item[0])
        anchor = TF.to_tensor(anchor)
        pos = TF.to_tensor(pos)
        neg = TF.to_tensor(neg)
        return anchor, pos, neg

    # returns Tensor
    def test_getitem(self, index):
        test_item = self.test_data[index]
        anchor, pos, neg = self.getTriplet(test_item[0])
        anchor = TF.to_tensor(anchor)
        pos = TF.to_tensor(pos)
        neg = TF.to_tensor(neg)
        return anchor, pos, neg

    def train_len(self):
        return len(self.train_data)

    def test_len(self):
        return len(self.test_data)

    # returns: List[Tuple[Tensor(anchor), Tensor(positive), Tensor(negative)]]
    def get_train_data_list(self):
        final_train_data = []
        for index in range(self.train_len()):
            final_train_data.append(self.train_getitem(index))
        return final_train_data

    # returns: List[Tuple[Tensor(anchor), Tensor(positive), Tensor(negative)]]
    def get_test_data_list(self):
        final_test_data = []
        for index in range(self.test_len()):
            final_test_data.append(self.test_getitem(index))
        return final_test_data

why kaggle train data length is 3486???

In [ ]:
m = MyDataset('Kaggle')
m.get_train_data_list()

In [ ]:
# config
batch_size = 2
test_batch_size = 1
num_workers = 0
validation_size = 0.1
validation_iteration = 5000
n_epochs = 10
learning_rate = 0.0001
weight_decay=1e-5
image_size = 512
device = torch.device('cuda:0')

In [ ]:
# Data
d = MyDataset(dataset)
train_data = d.get_train_data_list()
# test_data = d.get_test_data_list()

# train_data to train and validation:
indices = list(range(len(train_data)))
np.random.shuffle(indices) # necessary?
split = int(np.floor(validation_size * len(train_data))) # 3 validations
train_index, validation_index = indices[split:], indices[:split]
# print(train_index)

train_sampler = SubsetRandomSampler(train_index)
validation_sampler = SubsetRandomSampler(validation_index)

# loaders
train_loader = torch.utils.data.DataLoader(train_data,
                                           batch_size=batch_size,
                                           sampler=train_sampler,
                                           num_workers=num_workers)
validation_loader = torch.utils.data.DataLoader(train_data,
                                                batch_size=batch_size,
                                                sampler=validation_sampler,
                                                num_workers=num_workers)
# test_loader = torch.utils.data.DataLoader(test_data,
                                          # batch_size=batch_size,
                                          # num_workers=num_workers)

Load with label, for test

In [ ]:
class Test_Dataset(Dataset):

    def __init__(self, dataset='Kaggle', train=True):
        self.TEST_DATA_PATH = "/content/Data/KaggleTrainingData"
        self.TEST_LABEL_PATH = "/content/Data/KaggleTrainingData/Labels"
        self.test_data = torchvision.datasets.ImageFolder(root=self.TEST_DATA_PATH)
        self.test_label = torchvision.datasets.ImageFolder(root=self.TEST_LABEL_PATH)

    def test_getitem(self, index):
        resize = transforms.Resize(size=(512, 512))
        item = self.test_data[index]
        label = self.test_label[index]
        item = resize(item[0])
        label = resize(label[0])
        item = TF.to_tensor(item)
        label = TF.to_tensor(label)
        return item, label

    def __len__(self):
        # return len(self.test_data)
        return 10 ##################### for now

    # returns: 
    def get_test_list(self):
        final_test_data = []
        final_test_label = []
        for index in range(self.__len__()):
            item, label = self.test_getitem(index)
            final_test_data.append([item,label])
            # final_test_label.append(label)
        return final_test_data #, final_test_label

In [ ]:
# Data
dataset = 'Kaggle'
td = Test_Dataset(dataset)
# test_data, test_label = td.get_test_list()
test_data = td.get_test_list()
test_loader = torch.utils.data.DataLoader(test_data,
                                          batch_size=test_batch_size,
                                          num_workers=num_workers)

## Auto-Encoder

Unet

In [ ]:
class DoubleConv(nn.Module):
    """(convolution => [BN] => ReLU) * 2"""

    def __init__(self, in_channels, out_channels, mid_channels=None):
        super().__init__()
        if not mid_channels:
            mid_channels = out_channels
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)


class Down(nn.Module):
    """Downscaling with maxpool then double conv"""

    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels)
        )

    def forward(self, x):
        return self.maxpool_conv(x)


class Up(nn.Module):
    """Upscaling then double conv"""

    def __init__(self, in_channels, out_channels, bilinear=True):
        super().__init__()

        # if bilinear, use the normal convolutions to reduce the number of channels
        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
            self.conv = DoubleConv(in_channels, out_channels, in_channels // 2)
        else:
            self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
            self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1, x2):
        x1 = self.up(x1)
        # input is CHW
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]

        x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2,
                        diffY // 2, diffY - diffY // 2])
        # if you have padding issues, see
        # https://github.com/HaiyongJiang/U-Net-Pytorch-Unstructured-Buggy/commit/0e854509c2cea854e247a9c615f175f76fbb2e3a
        # https://github.com/xiaopeng-liao/Pytorch-UNet/commit/8ebac70e633bac59fc22bb5195e513d5832fb3bd
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)


class OutConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(OutConv, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=1)

    def forward(self, x):
        return self.conv(x)


# class UNet(nn.Module):
#     def __init__(self, n_channels=3, n_classes=3, base_channel=64, bilinear=True):
#         super(UNet, self).__init__()
#         self.n_channels = n_channels
#         self.n_classes = n_classes
#         self.bilinear = bilinear

#         self.inc = DoubleConv(n_channels, base_channel)
#         self.down1 = Down(base_channel, 2 * base_channel)
#         self.down2 = Down(2 * base_channel, 4 * base_channel)
#         self.down3 = Down(4 * base_channel, 8 * base_channel)
#         factor = 2 if bilinear else 1
#         self.down4 = Down(8 * base_channel, 16 * base_channel // factor)
#         self.up1 = Up(16 * base_channel, 8 * base_channel // factor, bilinear)
#         self.up2 = Up(8 * base_channel, 4 * base_channel // factor, bilinear)
#         self.up3 = Up(4 * base_channel, 2 * base_channel // factor, bilinear)
#         self.up4 = Up(2 * base_channel, base_channel, bilinear)
#         self.outc = OutConv(base_channel, n_classes)

#     def forward(self, x):
#         x1 = self.inc(x)
#         x2 = self.down1(x1)
#         x3 = self.down2(x2)
#         x4 = self.down3(x3)
#         x5 = self.down4(x4)
#         x = self.up1(x5, x4)
#         # shape = x5.shape
#         # x = self.up1(x5.view(-1, shape[1], shape[2], shape[3]), x4)
#         x = self.up2(x, x3)
#         x = self.up3(x, x2)
#         x = self.up4(x, x1)
#         x = self.outc(x)
#         # x = nn.Linear(x, 1000) # add this line myself
#         return x



class Unet_encoder(nn.Module):
      def __init__(self, n_channels=3, n_classes=3, base_channel=64, bilinear=True):
        super(Unet_encoder, self).__init__()
        self.n_channels = n_channels
        self.n_classes = n_classes
        self.bilinear = bilinear

        self.inc = DoubleConv(n_channels, base_channel)
        self.down1 = Down(base_channel, 2 * base_channel)
        self.down2 = Down(2 * base_channel, 4 * base_channel)
        self.down3 = Down(4 * base_channel, 8 * base_channel)
        factor = 2 if bilinear else 1
        self.down4 = Down(8 * base_channel, 16 * base_channel // factor)

      def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        return x5, [x1,x2,x3,x4]


class Unet_decoder(nn.Module):
    def __init__(self, n_channels=3, n_classes=3, base_channel=64, encoder_weights=[], bilinear=True):
        super(Unet_decoder, self).__init__()
        self.n_channels = n_channels
        self.n_classes = n_classes
        self.bilinear = bilinear
        self.x1, self.x2, self.x3, self.x4 = encoder_weights

        factor = 2 if bilinear else 1
        self.up1 = Up(16 * base_channel, 8 * base_channel // factor, bilinear)
        self.up2 = Up(8 * base_channel, 4 * base_channel // factor, bilinear)
        self.up3 = Up(4 * base_channel, 2 * base_channel // factor, bilinear)
        self.up4 = Up(2 * base_channel, base_channel, bilinear)
        self.outc = OutConv(base_channel, n_classes)
    
    def forward(self, x5):
        x = self.up1(x5, self.x4)
        # shape = x5.shape
        # x = self.up1(x5.view(-1, shape[1], shape[2], shape[3]), x4)
        x = self.up2(x, self.x3)
        x = self.up3(x, self.x2)
        x = self.up4(x, self.x1)
        x = self.outc(x)
        return x

### Display Architecture

Print Architucture Summery

In [ ]:
from torchsummary import summary
summary(unet,(3,768,768))
# print(unet)

To Print Graphical Architecture

In [ ]:
# !pip install graphviz
# !pip install torchviz

from torchviz import make_dot, make_dot_from_trace

x = torch.randn(1, 3, 768, 768).to(device)
make_dot(unet(x), params=dict(unet.named_parameters()))

### Train

Loss

In [ ]:
class TripletNet(nn.Module):
    def __init__(self, Unet_encoder):
        super(TripletNet, self).__init__()
        self.Unet_encoder = Unet_encoder
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(294912, 256)     
        self.fc2 = nn.Linear(294912, 256)  
        self.fc3 = nn.Linear(294912, 256)                  
                                
    def forward(self, i1, i2, i3):
        i1 = self.Unet_encoder(i1)[0]
        i1 = self.flatten(i1)
        E1 = self.fc1(i1)
        i2 = self.Unet_encoder(i2)[0]
        i2 = self.flatten(i2)
        E2 = self.fc1(i2)
        i3 = self.Unet_encoder(i3)[0]
        i3 = self.flatten(i3)
        E3 = self.fc1(i3)
        return E1,E2,E3

def tripletLoss(margin, anchor, positive, negative, size_average=False):
      distance_positive = F.pairwise_distance(anchor , positive)
      distance_negative = F.pairwise_distance(anchor , negative)
      losses = distance_positive - distance_negative + margin
      loss = None
      if size_average :
        loss = torch.max(torch.FloatTensor([0]).to(device) , torch.mean(losses))
      else :
        loss = torch.max(torch.FloatTensor([0]).to(device) , torch.sum(losses))
      return loss


class CountRankingModel(nn.Module):

    def __init__(self):
        super(CountRankingModel, self).__init__()
        self.conv = nn.Conv1d(1 , 1, kernel_size=256, stride=1, padding=0)


    def forward(self, positive, negative):
        positive = positive.view(positive.size()[0],1, 256)
        negative = negative.view(negative.size()[0],1, 256)
        p = self.conv(positive)
        p = p.view(p.size()[0], -1)
        n = self.conv(negative)
        n = n.view(n.size()[0], -1)
        return p , n

def countLoss(model , margin , z_postive , z_negative ,  size_average=True):
      out_p , out_n = model(z_postive, z_negative)
      losses = F.relu (out_p - out_n + margin)
      loss = None
      if size_average :
        loss = torch.max(torch.FloatTensor([0]).to(device) , torch.mean(losses))
      else :
        loss = torch.max(torch.FloatTensor([0]).to(device) , torch.sum(losses))
      return loss 

Display

In [ ]:
def plot_loss(train, scaleWise, countRank):
    y = [train, scaleWise, countRank]
    labels=["Train Loss","ScaleWise Loss","Count Ranking Loss"]

    for i in range(len(y)):
        plt.plot(y[i],label=labels[i])

    plt.legend()
    plt.show()

def show_images(images: list) -> None:
    n: int = len(images)
    f = plt.figure()
    for i in range(n):
        # Debug, plot figure
        ax = f.add_subplot(1, n, i + 1)
        plt.imshow(images[i])
    plt.show(block=True)

In [ ]:
triplet_model = TripletNet(Unet_encoder(3,3,64))
model = nn.DataParallel(triplet_model)
c_model = CountRankingModel().to(device)

Alpha = 0.01

# out = Unet_encoder(3,3,64)
# out = out(torch.randn(1,3,768,768))

# print(out.size())

# out1,out2,out3= model(torch.randn(1,3,768,768).to(device),torch.randn(1,3,768,768).to(device),torch.randn(1,3,768,768).to(device))

to_print_trans = transforms.ToPILImage()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

validation_loss_min = np.Inf
train_loss_values = []
train_triplet_loss_values = []
train_count_loss_values = []
a_list = []
p_list = []
n_list = []

n_epochs = 10
for epoch in range(n_epochs):
    print('=========================================================')
    print('Epoch: {}'.format(epoch + 1))
    train_loss = 0
    model.train()
    t_loss_total = 0
    c_loss_total = 0
    for iteration, (anchor, positive, negative) in enumerate(train_loader):
        print_triplet = (to_print_trans(anchor[0, :, :, :].to('cpu')), to_print_trans(positive[0, :, :, :].to('cpu')), to_print_trans(negative[0, :, :, :].to('cpu')))
        # print('-- iteration: {}'.format(iteration))
        optimizer.zero_grad()
        # compute output
        out_a, out_p, out_n = model(anchor, positive, negative) 
        # out_a, a_list = out_a
        # out_p, p_list = out_p
        # out_n, n_list = out_n
        t_loss = tripletLoss(1, out_a, out_p, out_n , False) 
        c_loss = countLoss(c_model , 1 , out_p , out_n , False)
        loss = t_loss + Alpha*c_loss
        loss.sum().backward()
        optimizer.step()

        # update running train loss
        train_loss += loss.item()
        
        if iteration % validation_iteration == 0: # what validation_iteration is good???
            with torch.no_grad():
                is_training = model.training
                validation_loss = 0
                model.eval()

                for anchor, positive, negative in validation_loader:
                  out_a, out_p, out_n = model(anchor , positive, negative)
                  # out_a, _ = out_a
                  # out_p, _ = out_p
                  # out_n, _ = out_n
                  v_t_loss = tripletLoss(1, out_a, out_p, out_n)
                  v_c_loss = countLoss(c_model , 1 , out_p , out_n , False) 
                  loss = v_t_loss + Alpha*v_c_loss
                  # print("total_val_loss : {:.2f} , ScaleWizeLoss : {:.2f} CountLoss : {:.2f}".format( float(loss) , float(v_t_loss) ,float(v_c_loss) ))
                  # update running validation loss
                  validation_loss += loss.item()

                validation_loss = validation_loss / len(validation_loader.sampler)

                # save model if validation loss has decreased
                if validation_loss <= validation_loss_min:
                    print('\tValidation loss: {:.2f}'.format(validation_loss))
                    torch.save(model.state_dict(), 'model.pt')
                    print('Model Saved')
                    validation_loss_min = validation_loss

                model.train(mode=is_training)
        t_loss_total += t_loss
        c_loss_total += Alpha*c_loss
    train_loss = train_loss / len(train_loader.sampler)
    train_loss_values.append(train_loss)
    t_loss_total = float(t_loss_total)/len(train_loader.sampler)
    c_loss_total = float(c_loss_total)/len(train_loader.sampler)
    train_triplet_loss_values.append(t_loss_total)
    train_count_loss_values.append(c_loss_total)
    show_images(print_triplet)
    print('Epoch: {} \tTraining Loss: {:.3f} \n\t\tScaleWizeLoss : {:.3f}\n\t\tCountLoss : {:.3f}'.format(epoch + 1, train_loss , t_loss_total , c_loss_total))

plot_loss(train_loss_values, train_triplet_loss_values, train_count_loss_values)

In [ ]:
dec_model = Unet_decoder(3,3,64,a_list).to(device)
dec_model.eval()

test_loss = 0.0
criterion = nn.MSELoss()


for img in test_loader:
    image, target = img
    ### should be implemented cleaner :/
    out, _, _ = model(image, image, image)
    out, _ = out
    # print(out.size())
    final_out = dec_model(out.to(device))
    print_result = (to_print_trans(final_out[0, :, :, :].to('cpu')), to_print_trans(target[0, :, :, :].to('cpu')))
    # loss = criterion(final_out, target)
    # test_loss += loss.item()

show_images(print_result)
# test_loss = test_loss / len(train_loader)

print('\tTest Loss: {:.10f}'.format(test_loss))


# **TRAINING WITH DIFFERENT NETWORKS**

# Custom Architucture

In [ ]:
class EmbeddingNet(nn.Module):
    def __init__(self):
        super(EmbeddingNet, self).__init__()
        self.convNet = nn.Sequential(nn.Conv2d(3, 8, kernel_size=5, stride=3, padding=1),
                                     nn.ReLU(),
                                     nn.Conv2d(8, 16, kernel_size=3, stride=2, padding=1),
                                     nn.ReLU(),
                                     nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1),
                                     nn.ReLU(),
                                     nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
                                     nn.ReLU()
                                     )
        # this part has problems
        self.fc = nn.Sequential(nn.Linear(65536, 1024), 
                                nn.ReLU(),
                                nn.Linear(1024, 256),
                                nn.ReLU()
                                )

    def forward(self, x):
        output = self.convNet(x)
        # print(output.shape)
        output = output.view(output.size()[0], -1)
        # print(output.shape)
        output = self.fc(output)
        # print(output.shape)
        return output

    def get_embedding(self, x):
        return self.forward(x)

In [ ]:
class TripletNet(nn.Module):
    def __init__(self, embeddingNet):
        super(TripletNet, self).__init__()
        self.embeddingNet = embeddingNet
        self.fc1 = nn.Linear(256, 16)     
        self.fc2 = nn.Linear(256, 16)  
        self.fc3 = nn.Linear(256, 16)                  
                                
    def forward(self, i1, i2, i3):
        i1 = self.embeddingNet(i1)
        E1 = self.fc1(i1)
        i2 = self.embeddingNet(i2)
        E2 = self.fc1(i2)
        i3 = self.embeddingNet(i3)
        E3 = self.fc1(i3)
        return E1, E2, E3

def tripletLoss(margin, anchor, positive, negative, size_average=False):
      distance_positive = F.pairwise_distance(anchor , positive)
      distance_negative = F.pairwise_distance(anchor , negative)
      losses = distance_positive - distance_negative + margin
      loss = None
      if size_average :
        loss = torch.max(torch.FloatTensor([0]).to(device) , torch.mean(losses))
      else :
        loss = torch.max(torch.FloatTensor([0]).to(device) , torch.sum(losses))
      return loss

class CountRankingModel(nn.Module):

    def __init__(self):
        super(CountRankingModel, self).__init__()
        self.conv = nn.Conv1d(1 , 1, kernel_size=16, stride=1, padding=0)


    def forward(self, positive, negative):
        positive = positive.view(positive.size()[0],1, 16)
        negative = negative.view(negative.size()[0],1, 16)
        p = self.conv(positive)
        p = p.view(p.size()[0], -1)
        n = self.conv(negative)
        n = n.view(n.size()[0], -1)
        return p , n

def countLoss(model , margin , z_postive , z_negative ,  size_average=True):
      out_p , out_n = model(z_postive, z_negative)
      losses = F.relu (out_p - out_n + margin)
      loss = None
      if size_average :
        loss = torch.max(torch.FloatTensor([0]).to(device) , torch.mean(losses))
      else :
        loss = torch.max(torch.FloatTensor([0]).to(device) , torch.sum(losses))
      return loss 

Train With Custom Architecture

In [ ]:
%matplotlib inline
embedding_model = EmbeddingNet()
triplet_model = TripletNet(embedding_model)
model = nn.DataParallel(triplet_model)
# model.to(device)
c_model = CountRankingModel().to(device)


to_print_trans = transforms.ToPILImage()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

validation_loss_min = np.Inf
train_loss_values = []

for epoch in range(n_epochs):
    train_loss = 0
    model.train()

    for iteration, (anchor, positive, negative) in enumerate(train_loader):
        print_triplet = (to_print_trans(anchor[0, :, :, :].to('cpu')), to_print_trans(positive[0, :, :, :].to('cpu')), to_print_trans(negative[0, :, :, :].to('cpu')))
        
        optimizer.zero_grad()
         # compute output
        out_a, out_p, out_n = model(anchor, positive, negative) 
        t_loss = tripletLoss(1, out_a, out_p, out_n , False) 
        c_loss = countLoss(c_model , 1 , out_p , out_n , False)
        loss = t_loss + c_loss
        loss.sum().backward()
        optimizer.step()

        # update running train loss
        train_loss += loss.item()

        if iteration % validation_iteration == 0: # what validation_iteration is good???
            is_training = model.training
            validation_loss = 0
            model.eval()  # preparing model for evaluation
            for anchor, positive, negative in validation_loader:
               out_a, out_p, out_n = model(anchor , positive, negative)
               t_loss = tripletLoss(1, out_a, out_p, out_n)
               c_loss = countLoss(c_model , 1 , out_p , out_n , False) 
               loss = t_loss + c_loss
               print("total_val_loss : " , float(loss) , " scaleWizeLoss : " , float(t_loss) , " CountLoss : " , float(c_loss))
               # update running validation loss
               validation_loss += loss.item()

            validation_loss = validation_loss / len(validation_loader.sampler)

            # save model if validation loss has decreased
            if validation_loss <= validation_loss_min:
                print('\tValidation loss: {:.10f}'.format(validation_loss))
                torch.save(model.state_dict(), 'model.pt')
                validation_loss_min = validation_loss

            model.train(mode=is_training)

    train_loss = train_loss / len(train_loader.sampler)
    train_loss_values.append(train_loss)
    show_images(print_triplet)
    print('Epoch: {} \tTraining Loss: {:.10f}'.format(epoch + 1, train_loss))
plt.plot(train_loss_values)
plt.xlabel('Epoch')  
# plt.ylabel('loss')
plt.title("Train Loss") 



Test With Custom Architecture

In [ ]:
#load the last model
# model = EmbeddingNet()
# model.load_state_dict(torch.load('model.pt'))
# #print(model)

#testing
test_loss = 0.0

model.eval()

for anchor, positive, negative in test_loader:
    out_a, out_p, out_n = model(anchor, positive,negative)
    t_loss = tripletLoss(1, out_a, out_p, out_n , False) 
    c_loss = countLoss(c_model , 1 , out_p , out_n , False)
    loss = t_loss + c_loss
    test_loss += loss.item()

test_loss = test_loss / len(train_loader)

print('\tTest Loss: {:.10f}'.format(test_loss))


# ResNet-101

In [ ]:
Res_model = torch.hub.load('pytorch/vision:v0.6.0', 'resnet101', pretrained=True)

### Display Architecture

Print Architucture Summery

In [ ]:
from torchsummary import summary
summary(Res_model.to(device), (3, 768, 768))
# print(Res_model)

To Print Graphical Architecture

In [ ]:
# !pip install graphviz
# !pip install torchviz

from torchviz import make_dot, make_dot_from_trace

x = torch.randn(1, 3, 768, 768).to(device)
make_dot(Res_model(x), params=dict(Res_model.named_parameters()))

### Train

In [ ]:
#ResNet

class TripletNet(nn.Module):
    def __init__(self, Res_model):
        super(TripletNet, self).__init__()
        self.Res_model = Res_model
        self.fc1 = nn.Linear(1000, 128)     
        self.fc2 = nn.Linear(1000, 128)  
        self.fc3 = nn.Linear(1000, 128)                  
                                
    def forward(self, i1, i2, i3):
        i1 = self.Res_model(i1)
        E1 = self.fc1(i1)
        i2 = self.Res_model(i2)
        E2 = self.fc1(i2)
        i3 = self.Res_model(i3)
        E3 = self.fc1(i3)
        return E1, E2, E3

def tripletLoss(margin, anchor, positive, negative, size_average=False):
      distance_positive = F.pairwise_distance(anchor , positive)
      distance_negative = F.pairwise_distance(anchor , negative)
      losses = distance_positive - distance_negative + margin
      loss = None
      if size_average :
        loss = torch.max(torch.FloatTensor([0]).to(device) , torch.mean(losses))
      else :
        loss = torch.max(torch.FloatTensor([0]).to(device) , torch.sum(losses))
      return loss


class CountRankingModel(nn.Module):

    def __init__(self):
        super(CountRankingModel, self).__init__()
        self.conv = nn.Conv1d(1 , 1, kernel_size=128, stride=1, padding=0)


    def forward(self, positive, negative):
        positive = positive.view(positive.size()[0],1, 128)
        negative = negative.view(negative.size()[0],1, 128)
        p = self.conv(positive)
        p = p.view(p.size()[0], -1)
        n = self.conv(negative)
        n = n.view(n.size()[0], -1)
        return p , n

def countLoss(model , margin , z_postive , z_negative ,  size_average=True):
      out_p , out_n = model(z_postive, z_negative)
      losses = F.relu (out_p - out_n + margin)
      loss = None
      if size_average :
        loss = torch.max(torch.FloatTensor([0]).to(device) , torch.mean(losses))
      else :
        loss = torch.max(torch.FloatTensor([0]).to(device) , torch.sum(losses))
      return loss 

Train With ResNet-101 Architecture

In [ ]:
torch.cuda.empty_cache()

In [ ]:
%matplotlib inline

triplet_model = TripletNet(Res_model)
model = nn.DataParallel(triplet_model)
c_model = CountRankingModel().to(device)

# out = Res_model(torch.randn(1,3,768,768).to(device))
# out1,out2,out3= model(torch.randn(1,3,768,768).to(device),torch.randn(1,3,768,768).to(device),torch.randn(1,3,768,768).to(device))

to_print_trans = transforms.ToPILImage()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

validation_loss_min = np.Inf
train_loss_values = []

for epoch in range(n_epochs):
    train_loss = 0
    model.train()

    for iteration, (anchor, positive, negative) in enumerate(train_loader):
        print_triplet = (to_print_trans(anchor[0, :, :, :].to('cpu')), to_print_trans(positive[0, :, :, :].to('cpu')), to_print_trans(negative[0, :, :, :].to('cpu')))
        
        optimizer.zero_grad()
         # compute output
        out_a, out_p, out_n = model(anchor, positive, negative) 
        t_loss = tripletLoss(1, out_a, out_p, out_n , False) 
        c_loss = countLoss(c_model , 1 , out_p , out_n , False)
        loss = t_loss + c_loss
        loss.sum().backward()
        optimizer.step()

        # update running train loss
        train_loss += loss.item()
        
        if iteration % validation_iteration == 0: # what validation_iteration is good???
            with torch.no_grad():
                is_training = model.training
                validation_loss = 0
                model.eval()

                for anchor, positive, negative in validation_loader:
                  out_a, out_p, out_n = model(anchor , positive, negative)
                  v_t_loss = tripletLoss(1, out_a, out_p, out_n)
                  v_c_loss = countLoss(c_model , 1 , out_p , out_n , False) 
                  loss = v_t_loss + v_c_loss
                  print("total_val_loss : " , float(loss) , " ScaleWizeLoss : " , float(v_t_loss) , " CountLoss : " , float(v_c_loss))
                  # update running validation loss
                  validation_loss += loss.item()

                validation_loss = validation_loss / len(validation_loader.sampler)

                # save model if validation loss has decreased
                if validation_loss <= validation_loss_min:
                    print('\tValidation loss: {:.10f}'.format(validation_loss))
                    torch.save(model.state_dict(), 'model.pt')
                    validation_loss_min = validation_loss

                model.train(mode=is_training)

    train_loss = train_loss / len(train_loader.sampler)
    train_loss_values.append(train_loss)
    show_images(print_triplet)
    print('Epoch: {} \tTraining Loss: {:.10f}'.format(epoch + 1, train_loss), "\n\t\tScaleWizeLoss : " , float(t_loss) , "\n\t\tCountLoss : " , float(c_loss))
plt.plot(train_loss_values)
plt.xlabel('Epoch')  
# plt.ylabel('loss')
plt.title("Train Loss") 


In [ ]:
#testing
test_loss = 0.0

model.eval()

for anchor, positive, negative in test_loader:
    out_a, out_p, out_n = model(anchor, positive,negative)
    t_loss = tripletLoss(1, out_a, out_p, out_n , False) 
    c_loss = countLoss(c_model , 1 , out_p , out_n , False)
    loss = t_loss + c_loss
    test_loss += loss.item()

test_loss = test_loss / len(train_loader)

print('\tTest Loss: {:.10f}'.format(test_loss))
